# Xay dung va huan luyen mang DNN bang TensorFlow/Keras

**Muc tieu bai hoc:**
- Hieu kien truc cua mot mang no-ron nhieu lop (Deep Neural Network - DNN): cac lop Dense, ham
  kich hoat (activation), vi sao mang can "sau" va vi sao can phi tuyen.
- Hieu ban chat cua 4 buoc huan luyen mot mang no-ron: **forward propagation** (tinh du doan),
  **cost function** (do sai so), **backward propagation** (tinh gradient), **gradient descent**
  (cap nhat trong so) - va TensorFlow/Keras tu dong hoa nhung buoc nay nhu the nao.
- Xay dung, huan luyen va danh gia mot mang DNN hoan chinh bang `tf.keras` tren du lieu y te that
  (du doan benh tieu duong).

In [ ]:
# --- Google Colab setup: bo qua tu dong neu chay Jupyter local ---
import sys, os

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"  # tat toi uu oneDNN de dam bao ket qua giong het nhau
# giua cac lan chay - phai dat TRUOC khi 'import tensorflow' o cell sau (oneDNN co the doi thu
# tu tinh toan tren nhieu luong CPU, gay lech ket qua nho o vai chu so thap phan cuoi giua cac lan)

if "google.colab" in sys.modules:
    REPO_DIR = "/content/AI-Teaching-Labs"
    if not os.path.isdir(REPO_DIR):
        !git clone --depth 1 https://github.com/NonomiyaIzumi/AI-Teaching-Labs.git {REPO_DIR}
    NOTEBOOK_DIR = f"{REPO_DIR}/Deep Learning/Module 1/keras-tensorflow/01_Kien-truc-DNN-va-Backpropagation"
    os.chdir(NOTEBOOK_DIR)
    print("Da chuyen working directory sang:", os.getcwd())
else:
    print("Khong phai Colab - dung working directory hien tai:", os.getcwd())
    print("Neu chay Jupyter local: tu thu muc keras-tensorflow/, chay 'uv sync' truoc.")

## 1. Mang no-ron nhieu lop (Deep Neural Network) la gi?

Mot mang DNN gom nhieu **lop** (layer) xep chong len nhau. Moi lop nhan dau vao tu lop truoc, ap
dung mot phep bien doi tuyen tinh roi mot ham phi tuyen, va truyen ket qua sang lop sau:

$$Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}, \qquad A^{[l]} = g^{[l]}(Z^{[l]})$$

trong do `W^[l]`, `b^[l]` la trong so/bias cua lop `l`, va `g^[l]` la ham kich hoat (activation).
Voi bai toan **phan loai nhi phan** (0/1), kien truc pho bien la:

$$\text{Input} \;\to\; [\text{Dense} \to \text{ReLU}] \times (L-1) \;\to\; \text{Dense} \to \text{Sigmoid} \;\to\; \hat{y}$$

- **ReLU** (`g(z) = max(0, z)`) dung cho cac hidden layer - re, khong bi vanishing gradient nhu
  Sigmoid/Tanh khi `z` lon.
- **Sigmoid** (`g(z) = 1/(1+e^{-z})`) dung cho lop cuoi - nen dau ra ve khoang `(0, 1)`, doc duoc
  nhu mot xac suat `P(y=1 | x)`.
- **Vi sao can nhieu lop (sau) va can phi tuyen?** Neu bo activation phi tuyen, nhieu lop Dense
  chong len nhau chi tuong duong **mot** phep bien doi tuyen tinh duy nhat (tich cac ma tran vAn la
  mot ma tran) - mang se khong the hoc duoc cac quan he phuc tap, phi tuyen giua dac trung dau vao
  va nhan. Chinh ham kich hoat phi tuyen giup mang "sau" thuc su manh hon mang "nong" (1 lop).

## 2. Bon buoc huan luyen mot mang no-ron

Du ban tu viet tay (NumPy) hay dung framework (Keras), qua trinh huan luyen mot mang no-ron bang
gradient descent luon gom 4 buoc lap di lap lai:

1. **Forward propagation** - dua du lieu `X` qua tung lop theo cong thuc o muc 1, thu duoc du doan
   `\hat{y} = A^{[L]}`.
2. **Cost function** - do "mang sai bao nhieu" bang mot con so duy nhat. Voi phan loai nhi phan,
   dung **binary cross-entropy**:
   $$J = -\frac{1}{m}\sum_{i=1}^m \Big[y^{(i)}\log(\hat y^{(i)}) + (1-y^{(i)})\log(1-\hat y^{(i)})\Big]$$
3. **Backward propagation** - tinh dao ham `dJ/dW^[l]`, `dJ/db^[l]` cho MOI tham so, bang quy tac
   chuoi (chain rule) lan nguoc tu lop cuoi ve lop dau. Day la buoc de nham lan/de sai nhat khi tu
   viet tay (mot ly do bai gradient checking o phan sau ton tai).
4. **Cap nhat tham so (gradient descent)** - dich chuyen tung tham so nguoc huong gradient mot
   luong `learning_rate`:
   $$W^{[l]} := W^{[l]} - \alpha \, \frac{\partial J}{\partial W^{[l]}}$$

**Diem mau chot cua `tf.keras`:** ban van phai **tu thiet ke kien truc** (buoc 1 - so lop, so
neuron, activation nao), nhung buoc 2-4 (cost, backward, update) duoc Keras tu dong hoa hoan toan:
- `model.compile(loss=..., optimizer=...)` khai bao cost function va thuat toan cap nhat.
- `model.fit(X, y, epochs=...)` tu chay forward, tu tinh gradient bang **automatic differentiation**
  (khong xap xi nhu numerical gradient, ma tinh dung chinh xac theo chain rule - giong het buoc 3 o
  tren nhung khong can ban tu viet cong thuc), roi tu cap nhat tham so moi epoch.

## 3. Du lieu: Pima Indians Diabetes

Bo du lieu y te thuc te gom **768 benh nhan nu goc An Do (Pima)**, moi nguoi co **8 dac trung lam
sang** va 1 nhan nhi phan (co/khong mac tieu duong trong vong 5 nam):

| Dac trung | Y nghia |
|---|---|
| Pregnancies | So lan mang thai |
| Glucose | Nong do glucose trong huyet tuong (do sau 2h uong dung dich glucose) |
| BloodPressure | Huyet ap tam truong (mm Hg) |
| SkinThickness | Do day nep gap da co tay (mm) |
| Insulin | Nong do insulin huyet thanh sau 2h (mu U/ml) |
| BMI | Chi so khoi co the |
| DiabetesPedigreeFunction | Chi so di truyen (tien su gia dinh mac tieu duong) |
| Age | Tuoi |
| **Outcome** | **Nhan: 0 = khong mac tieu duong, 1 = mac tieu duong** |

**Van de thuc te can xu ly:** 5 cot (`Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI`)
khong the co gia tri **0** ve mat sinh ly - gia tri `0` o day thuc chat la du lieu bi thieu (khong
do duoc), khong phai gia tri that. Neu giu nguyen, mo hinh se "hoc" sai rang `0` la mot gia tri binh
thuong. Cach xu ly: **median imputation** - thay `0` bang trung vi (median) cua cot do, tinh **tren
tap train** roi ap dung lai cho tap test (tranh **data leakage** - de lo thong tin tu tap test vao
qua trinh xu ly).

Sau khi thay gia tri thieu, du lieu duoc **chuan hoa** (chuan hoa Z-score: tru trung binh, chia do
lech chuan - cung tinh tren tap train) truoc khi dua vao mang - giup gradient descent hoi tu on
dinh hon, vi cac dac trung nhu `Age` (hang chuc) va `DiabetesPedigreeFunction` (hang phan thap phan)
co thang do rat khac nhau.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import layers

from keras_utils import load_dataset, compile_and_train, evaluate_classification

tf.config.experimental.enable_op_determinism()  # dam bao ket qua giong het nhau giua cac lan chay
tf.random.set_seed(1)
np.random.seed(1)

train_X, train_Y, test_X, test_Y = load_dataset()
print("train_X:", train_X.shape, " train_Y:", train_Y.shape)
print("test_X: ", test_X.shape, " test_Y: ", test_Y.shape)

## 4. Xay dung kien truc mang bang `tf.keras.Sequential`

`keras.Sequential` cho phep khai bao mot mang bang cach liet ke tuan tu tung lop. Moi lop
`layers.Dense(units, activation=...)` tuong ung dung mot lop trong cong thuc o muc 1:

- `units` = so neuron cua lop do (kich thuoc `A^[l]`).
- `activation` = ham kich hoat `g^[l]` ap dung sau phep bien doi tuyen tinh.
- Keras **tu suy ra shape** cua `W^[l]`, `b^[l]` tu `units` cua lop truoc va lop hien tai (khong can
  ban tu khai bao shape thu cong).

Voi bai toan nay, ta dung kien truc `[8, 7, 1]`: 8 dac trung dau vao -> 1 hidden layer 7 neuron
(ReLU) -> 1 output neuron (Sigmoid).

### Bai tap - `build_model`

In [ ]:
def build_model(input_dim, hidden_units=7, seed=1):
    '''
    Xay dung mang [input_dim, hidden_units, 1]: Dense(hidden_units, ReLU) -> Dense(1, Sigmoid).
    (Tham so `seed` chi de dam bao ket qua tai lap duoc giua cac lan chay - khong phai trong tam
    bai hoc nay, xem chi tiet ve cach khoi tao trong so o notebook ve Vanishing/Exploding Gradient.)

    Returns: mot tf.keras.Model CHUA compile (chua co optimizer/loss).
    '''
    init = keras.initializers.GlorotUniform(seed=seed)
    # (~4 dong) dung keras.Sequential([...]) voi keras.Input + layers.Dense
    # model = keras.Sequential([
    #     keras.Input(shape=(input_dim,)),
    #     layers.Dense(hidden_units, activation="relu", kernel_initializer=init),
    #     layers.Dense(1, activation="sigmoid", kernel_initializer=init),
    # ])
    # YOUR CODE STARTS HERE
    pass
    # YOUR CODE ENDS HERE
    return model

In [ ]:
#@title Answer { display-mode: "form" }
def build_model(input_dim, hidden_units=7, seed=1):
    '''
    Xay dung mang [input_dim, hidden_units, 1]: Dense(hidden_units, ReLU) -> Dense(1, Sigmoid).
    (Tham so `seed` chi de dam bao ket qua tai lap duoc giua cac lan chay - khong phai trong tam
    bai hoc nay, xem chi tiet ve cach khoi tao trong so o notebook ve Vanishing/Exploding Gradient.)

    Returns: mot tf.keras.Model CHUA compile (chua co optimizer/loss).
    '''
    init = keras.initializers.GlorotUniform(seed=seed)
    # (~4 dong) dung keras.Sequential([...]) voi keras.Input + layers.Dense
    # YOUR CODE STARTS HERE
    model = keras.Sequential([
        keras.Input(shape=(input_dim,)),
        layers.Dense(hidden_units, activation="relu", kernel_initializer=init),
        layers.Dense(1, activation="sigmoid", kernel_initializer=init),
    ])
    # YOUR CODE ENDS HERE
    return model

In [ ]:
model = build_model(train_X.shape[0], hidden_units=7)
model.summary()

`model.summary()` in ra so tham so cua tung lop - doi chieu voi cong thuc: lop `Dense(7)` dau tien
co `8*7 + 7 = 63` tham so (`W1` shape `(7,8)` + `b1` shape `(7,)`), lop `Dense(1)` co `7*1 + 1 = 8`
tham so - dung bang tong so tham so `W1,b1,W2,b2` ma ban se tu tinh tay neu cai bang NumPy.

## 5. Bien dich va huan luyen mo hinh

`model.compile(...)` khai bao 3 thanh phan can cho buoc 2-4 o muc 2:

- `loss="binary_crossentropy"` - chinh la cong thuc cost function `J` o muc 2.
- `optimizer=keras.optimizers.SGD(learning_rate=...)` - thuat toan gradient descent se dung de cap
  nhat tham so (buoc 4). `SGD` la Gradient Descent "thuan" (khong momentum/adaptive learning rate).
- `metrics=["accuracy"]` - chi so theo doi them trong qua trinh train (khong dung de tinh gradient).

`model.fit(X, y, epochs=..., batch_size=...)` chay vong lap huan luyen: moi **epoch** la mot lan
di qua (mot phan) du lieu, thuc hien du forward -> tinh cost -> backward (tu dong qua autodiff) ->
cap nhat tham so. `batch_size=None` nghia la **full-batch gradient descent** - moi epoch dung toan
bo `m` vi du de tinh mot lan cap nhat duy nhat (giong dung mot "iteration" trong gradient descent
co ban).

Ta huan luyen `3000` epoch voi `learning_rate=0.5`.

In [ ]:
history = compile_and_train(
    model, train_X, train_Y,
    optimizer=keras.optimizers.SGD(learning_rate=0.5),
    epochs=3000, batch_size=None, verbose=0,
)
print(f"Cost cuoi cung (train):     {history.history['loss'][-1]:.4f}")
print(f"Accuracy cuoi cung (train): {history.history['accuracy'][-1]:.4f}")

## 6. Danh gia mo hinh tren tap test

Voi bai toan y te (du doan benh), **chi nhin Accuracy la khong du**: neu ~65% benh nhan trong tap
khong mac benh, mot mo hinh "ngu" luon du doan "khong mac benh" cung dat Accuracy ~65% ma khong hoc
duoc gi. Can nhin them:

- **Precision** = trong so du doan "co benh", bao nhieu % dung that su.
- **Recall** = trong so benh nhan THAT SU co benh, mo hinh phat hien dung bao nhieu % - **quan
  trong nhat trong y te**, vi bo sot mot ca benh that (false negative) thuong nguy hiem hon canh bao
  nham mot ca khong benh (false positive).
- **F1-score** = trung binh dieu hoa cua Precision va Recall.
- **Confusion matrix** - bang doi chieu du doan vs thuc te (4 o: TP, FP, FN, TN).
- **ROC curve / AUC** - do kha nang phan tach 2 lop cua mo hinh tren MOI nguong quyet dinh (khong
  chi nguong 0.5 mac dinh); AUC = 0.5 nghia la mo hinh doan ngau nhien, AUC = 1.0 la hoan hao.

In [ ]:
results = evaluate_classification(model, test_X, test_Y, "tf.keras Sequential [8, 7, 1] (test set)")

## 7. Ket qua & Phan tich

Voi kien truc `[8, 7, 1]`, `learning_rate=0.5`, `3000` epoch full-batch: **train Accuracy ~81%,
test Accuracy ~75%** (Precision ~71%, Recall ~50%, F1 ~0.59). Mo hinh hoc duoc tin hieu du doan
that su (ro rang tot hon baseline ~65% neu doan toan bo ve lop da so), nhung Recall con thap - mo
hinh con bo sot kha nhieu ca thuc su mac benh, mot huong cai thien tot cho phan **Bai tap mo rong**.
(Cac con so co the lech nhau vai phan tram giua cac lan chay du da co dinh seed - day la dac diem
binh thuong cua huan luyen bang gradient descent tren CPU nhieu luong, khong phai loi.)

**Diem mau chot can ghi nho:** kien truc mang, cost function, va thuat toan cap nhat tham so ban vua
dung (Dense + ReLU/Sigmoid, binary cross-entropy, gradient descent) chinh la nhung khai niem toan
hoc **can ban** cua moi mang no-ron, du ban tu cai dat tung buoc hay dung framework. Framework khong
"hoc gioi hon" - no chi tu dong hoa phan tinh dao ham (backward propagation) va vong lap cap nhat,
de ban tap trung vao **thiet ke kien truc va chon sieu tham so** thay vi tu suy dan cong thuc dao
ham cho tung lop.

## 8. Bai tap mo rong

1. **Tang do sau/do rong:** thu `hidden_units=20` hoac them mot lop `Dense` nua (vd `[8, 20, 10, 1]`)
   - Accuracy/Recall tren test set thay doi the nao? Mang sau hon co luon tot hon khong?
2. **Doi optimizer:** thay `keras.optimizers.SGD` bang `keras.optimizers.Adam` (giu nguyen kien
   truc/so epoch) - so sanh toc do hoi tu (nhin duong cost) va ket qua cuoi cung.
3. **Cai thien Recall:** thu class_weight (`model.fit(..., class_weight={0: 1, 1: w})` voi `w > 1`)
   de "phat" mo hinh nang hon khi bo sot ca mac benh - quan sat Recall co tang len khong, danh doi
   voi Precision the nao.
4. **Ve duong cost:** dung `history.history["loss"]` de ve bieu do cost theo epoch - cost co giam
   muot khong, co dau hieu overfitting (cost train giam nhung khong con phan anh dung tren tap test)
   khong?

## 9. Tai lieu tham khao

| Chu de | Lien ket |
|---|---|
| `tf.keras.Sequential` API | https://www.tensorflow.org/guide/keras/sequential_model |
| `Dense` layer | https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense |
| Binary cross-entropy loss | https://www.tensorflow.org/api_docs/python/tf/keras/losses/BinaryCrossentropy |
| Pima Indians Diabetes Dataset (nguon goc) | https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database |